In [5]:
import numpy as np
import matplotlib.pyplot as plt
import readfargo as fg
from makedustopacfortran import create_dustkapscatmat_file
import os
from radmc3dPy.analyze import readOpac
from scipy.optimize import bisect

Fast (Fortran90) Mie-scattering module could not be imported. Falling back to the slower Python version.


In [7]:
path = "/pfs/10/work/hd_cu284-work/fargo3d/outputs/fargo_fiducial_2gen"
dens = 1

au  = 1.49598e13     # Astronomical Unit       [cm]
pc  = 3.08572e18     # Parsec                  [cm]
ms  = 1.98892e33     # Solar mass              [g]
ts  = 5.78e3         # Solar temperature       [K]
ls  = 3.8525e33      # Solar luminosity        [erg/s]
rs  = 6.96e10        # Solar radius            [cm]
ss  = 5.6703e-5      # Stefan-Boltzmann const  [erg/cm^2/K^4/s]
kk  = 1.3807e-16     # Bolzmann's constant     [erg/K]
mp  = 1.6726e-24     # Mass of proton          [g]
GG  = 6.67408e-08    # Gravitational constant  [cm^3/g/s^2]
pi  = np.pi          # Pi

#
# Star parameters
#
mstar    = 1.08*ms #ms
rstar    = 1.418*rs # rs
tstar    = 4400 #K
pstar    = np.array([0.,0.,0.])

#
# Read a FARGO3D frame
#
# NOTE: Set the dir variable to the simulation you wish to import, and
#       the itime to the time snapshot you are interested in. The FARGO3D
#       simulations are in dimensionless units. By setting r0 in cm (see below)
#       and mstar in gram (see above), the simulation becomes "physical"
#       with units in CGS.
#
path    = path
itime    = 100
r0       = 57*au    # The radius corresponding to '1' in dimensionless units
fargo    = fg.frame(itime,rhodust=True,dir=path)
fargo.convert_to_cgs(mstar,r0)
#fargo.show(q=fargo.sigma_gas_cgs)
#fargo.show(q=fargo.sigma_dust_cgs[0])

#
# The 1-st dust component for RADMC-3D will follow the gas component
# of FARGO3D. We assume that these grains are small enough that they
# are well-mixed with the gas, at least near the midplane. We specify
# the dust-to-gas ratio for this fine-grained dust with the dtg_smalldust
# parameter.
#
# Any dust components in FARGO3D will be _in_addition_ to this first
# component
#
# NOTE: Without any small-grain component, the dust geometry will become
#       very geometrically thin, which means that very little starlight
#       will be captured, and the disk will become very cold. However,
#       if at least one of the dust components is vertically extended
#       (i.e. small grains), then more stellar radiation is captured
#       and the re-emission of this radiation will then also make the
#       large dust grains at the midplane warmer. Observations of most
#       protoplanetary disks show that most of them have some small
#       dust vertically extended above the midplane even though other
#       dust appears to be near the midplane (see e.g. IM Lup,
#       with the small grains seen in scattered light with VLT-SPHERE,
#       Avenhaus et al. 2018, ApJ 863, 44, and large grains seen with
#       ALMA, Huang et al., 2018, APJ 869, 43).
#
dtg_smalldust  = 1e-2        # Dust to gas ratio for small dust following the gas
#new_gasdens = np.outer(np.mean(fargo.sigma_gas_cgs, axis=1), np.ones(shape=len(fargo.sigma_gas_cgs[0])))
sigma_gas_2d   = fargo.sigma_gas_cgs
sigma_dust_2d  = []
sigma_dust_2d.append(sigma_gas_2d*dtg_smalldust*dens)  # First the gas-following small dust

#"""
#### THERE IS NO OTHER KINDS OF DUST IN THIS SIM ####
for d in fargo.sigma_dust_cgs:
    sigma_dust_2d.append(d)                       # Then the dynamic dust from FARGO3D multifluid
nrspec   = len(sigma_dust_2d)
print(f"{nrspec} dust species detected!")
#""" 


ri       = 0.5*(fargo.r_cgs[1:]+fargo.r_cgs[:-1])
ri       = np.hstack([2*ri[0]-ri[1],ri,2*ri[-1]-ri[-2]])
rc       = 0.5 * ( ri[:-1] + ri[1:] )#np.sqrt(ri[:-1] * ri[1:])
nr       = len(rc)   # Recompute nr, because of refinement at inner edge
r        = rc        # The radial grid of the analytic disk model (below)
phii     = 0.5*(fargo.phi[1:]+fargo.phi[:-1])+np.pi
phii     = np.hstack([2*phii[0]-phii[1],phii,2*phii[-1]-phii[-2]])
phic     = 0.5 * ( phii[:-1] + phii[1:] )
nphi     = len(phic)
r_2d,phi_2d = np.meshgrid(rc,phic,indexing='ij')



#
# Now make a simple analytical disk model roughly along the
# lines of Chiang & Goldreich (1997), but with just a single
# vertical layer and with a constant radiative incidence angle.
#

"""
How are the incidence angle and the flaring index/aspect ratio related?

The incidence angle is given by alpha = h/r -dh/dr (according to armitage's book). So it will depend on r for a flared disk!

Also, we're assuming here that the disk radiates as a black body and is vertially isothermal. (In particular there's no dust
in the way)

What is a reasonable value for flang?
It's not really self-consistent to choose a constant value for it . . .

"""

flang    = 0.05                        # The assumed constant radiative incidence angle
lstar    = 4*pi*rstar**2*ss*tstar**4   # Stellar luminosity
firr     = flang*lstar/(4*pi*r**2)     # Irradiative flux
tmid     = (firr/ss)**0.25             # Estimate of midplane temperature
cs       = np.sqrt(kk*tmid/(2.3*mp))   # Isothermal sound speed at midplane
omk      = np.sqrt(GG*mstar/r**3)      # The Kepler angular frequency
hp       = cs/omk                      # The pressure scale height
hpr      = hp/r                        # The dimensionless hp


1 dust species detected!


In [18]:
alpha_turb = 1e-3
nu_cgs = alpha_turb*(cs**2/omk)

D_cgs        = nu_cgs[114]   
print(D_cgs)

1667808092409351.2


In [13]:
np.interp(1,r/57/au,range(len(r)))

np.float64(114.16498757346076)